### La Fábrica Inteligente

**Contexto:** Eres el ingeniero a cargo de una flota de máquinas hidráulicas.

**Problema:** Si la máquina falla inesperadamente, la planta se detiene (pérdidas millonarias). Si haces mantenimiento preventivo demasiado pronto, gastas dinero innecesariamente.

**Datos:** Sensores en tiempo real (Temperatura, Velocidad de rotación, Torque, Desgaste de la herramienta).

**Objetivo:** Clasificación Binaria -> ¿Fallará la máquina en el próximo ciclo? (1: Sí, 0: No).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import files
files.upload()

data = pd.read_csv('ai4i2020.csv')

In [3]:
#data = pd.read_csv('data/ai4i2020.csv')
#data.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Dataset

### Descripción

**AI4I 2020 Predictive Maintenance Dataset:** Es un dataset sintético pero muy realista creado por el Fraunhofer Institute para simular procesos industriales de mecanizado (fresado).

### Las Variables Predictoras (Features - Los "Sensores")

Estas son las señales físicas que alimentan al algoritmo. 

- **Type (Tipo de Producto/Calidad):**
  
    - Calidad del producto que se está fabricando: L (Low/Bajo, 50% de los casos), M (Medium/Medio, 30%), H (High/Alto, 20%).
    - Nota: Las herramientas de calidad 'L' suelen romperse más. Hay que convertir esto a números (Encoding) para el modelo.
 
- **Air temperature [K] (Temperatura del Aire):** Es la temperatura ambiente en la fábrica (en Kelvin). Normalmente alrededor de 300K (27°C).
  
- **Process temperature [K] (Temperatura del Proceso):** Es la temperatura generada en la zona de corte.
    - Importante: Si la diferencia entre la temperatura de proceso y la del aire es muy baja, indica problemas de disipación de calor.
     
- **Rotational speed [rpm] (Velocidad de Rotación):** A cuántas revoluciones por minuto gira el husillo (cabezal) de la herramienta.
  
- **Torque [Nm] (Par Motor):** La fuerza de giro aplicada.
    - Física: Normalmente, a mayor velocidad, menor torque, y viceversa ($Potencia = Torque \times Velocidad$). Si ambos son altos a la vez, es una anomalía peligrosa (sobrecarga).
 
- **Tool wear [min] (Desgaste de Herramienta):** Tiempo acumulado que la herramienta ha estado cortando.
    - Intuición: Cuanto más alto es este número, más probable es que la herramienta se rompa por fatiga.

### Las Variables "Data Leakage

Estas columnas NO deben usarse como entrada (X) para predecir el fallo general, porque son el fallo en sí mismo desglosado por tipo. Si las incluyes, el modelo tendrá un 100% de acierto artificial.

Estas columnas son útiles para el diagnóstico post-mortem, pero no como sensores predictivos:

- **TWF (Tool Wear Failure):** Fallo por desgaste (la herramienta se usó demasiado).

- **HDF (Heat Dissipation Failure):** Fallo por calor (la diferencia de temperatura fue insuficiente).

- **PWF (Power Failure):** Fallo de potencia (Producto de Torque x Velocidad excesivo).

- **OSF (Overstrain Failure):** Fallo por sobrecarga (Torque o desgaste muy altos).

- **RNF (Random Failure):** Fallos aleatorios (simulan el caos del mundo real).

### Variables identificadoras

No aportan información física predictiva:

- **UDI:** Identificador único (1, 2, 3...). El modelo podría memorizar que "la fila 50 siempre falla", lo cual no sirve para datos nuevos.

- **Product ID:** Código de serie (M14860...). Demasiado específico, mejor usar solo Type

## Variable Objetivo

### Variable Objetivo (Target)

**Machine failure (Fallo de Máquina):**

- Tipo: Binaria (0 o 1).

- Significado: Indica si la máquina falló en ese ciclo concreto.

- **Valor 0:** Funcionamiento normal.

- **Valor 1:** La máquina se rompió o paró.